In [25]:
import pandas as pd

import mysql.connector
from config import DB_CONFIG

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

import joblib

In [6]:
# ------------------------------------------------------------
# 1. LOAD CLEANED DATA
# ------------------------------------------------------------
conn = mysql.connector.connect(**DB_CONFIG)
df = pd.read_sql('SELECT * FROM ecommerce_cleaned', con=conn)
conn.close()

C:\Users\Ekamjot Singh\AppData\Local\Temp\ipykernel_12964\156498892.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql('SELECT * FROM ecommerce_cleaned', con=conn)


In [7]:
categorical_cols = ['PreferredLoginDevice', 'CityTier', 'PreferredPaymentMode', 'Gender',
                    'PreferredOrderCat', 'SatisfactionScore','MaritalStatus']
for col in categorical_cols:
    df[col] = df[col].astype('category')

print(f"Data loaded: {df.shape}")

Data loaded: (5628, 20)


In [8]:
# ------------------------------------------------------------
# 2. FEATURE SELECTION
# ------------------------------------------------------------
df_model = df.drop(columns=['CustomerID', 'OrderAmountHikeFromlastYear'])

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

nominal_cols = ['Gender', 'MaritalStatus', 'PreferredLoginDevice',
                 'PreferredPaymentMode', 'PreferredOrderCat']
ordinal_cols = ['CityTier', 'SatisfactionScore']
numeric_cols = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp', 'CouponUsed',
                'OrderCount', 'DaySinceLastOrder', 'CashbackAmount',
                'NumberOfDeviceRegistered', 'NumberOfAddress']
binary_cols = ['Complain']

In [10]:
# ------------------------------------------------------------
# 3. TRAIN/TEST SPLIT
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)

# Sub-split training data for early stopping validation
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size = 0.15, stratify = y_train, random_state = 42)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (4502, 17) | Test: (1126, 17)


In [14]:
# ------------------------------------------------------------
# 4. PREPROCESSOR (tree-friendly, no scaling)
# ------------------------------------------------------------
preprocessor = ColumnTransformer(
    transformers = [
        ('nominal', OneHotEncoder(drop = "first", handle_unknown = "ignore"), nominal_cols),
        ('ordinal', "passthrough", ordinal_cols),
        ('numeric', "passthrough", numeric_cols),
        ('binary', "passthrough", binary_cols)
    ]
)

In [17]:
# ------------------------------------------------------------
# 5. FIND OPTIMAL TREE COUNT (n_estimators) VIA EARLY STOPPING
# ------------------------------------------------------------
preprocessor.fit(X_tr)
X_tr_transformed = preprocessor.transform(X_tr)
X_val_transformed = preprocessor.transform(X_val)

scale_pos_weight_es = (y_tr == 0).sum() / (y_tr == 1).sum()

es_model = XGBClassifier(
    n_estimators = 1000,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight_es,
    eval_metric='logloss',
    early_stopping_rounds=20,
    random_state=42
)

es_model.fit(X_tr_transformed, y_tr,
            eval_set = [(X_val_transformed, y_val)],
            verbose = False)

best_n_estimators = es_model.best_iteration
print(f"Optimal number of trees (via early stopping): {best_n_estimators}")

Optimal number of trees (via early stopping): 263


In [20]:
# ------------------------------------------------------------
# 6. BUILD FINAL PIPELINE AND FIT ON FULL TRAINING SET
# ------------------------------------------------------------
scale_pos_weight_full = (y_train == 0).sum() / (y_train == 1).sum()


final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        n_estimators = best_n_estimators,
        max_depth = 6,
        learning_rate = 0.1,
        scale_pos_weight = scale_pos_weight_full,
        eval_metric = "logloss",
        random_state = 42)
    )
])

final_pipeline.fit(X_train, y_train)
print("Final Model Trained.")

Final Model Trained.


In [24]:
# ------------------------------------------------------------
# 7. FINAL EVALUATION ON HELD-OUT TEST SET
# ------------------------------------------------------------
y_test_pred = final_pipeline.predict(X_test)
y_test_proba = final_pipeline.predict_proba(X_test)[:, 1]

print("\n=== FINAL MODEL — Held-Out Test Set Performance ===\n")
print(f"Accuracy:  {accuracy_score(y_test, y_test_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_test_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_test_pred):.3f}")
print(f"F1-score:  {f1_score(y_test, y_test_pred):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_test_proba):.3f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_test_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))


=== FINAL MODEL — Held-Out Test Set Performance ===

Accuracy:  0.992
Precision: 0.979
Recall:    0.974
F1-score:  0.976
ROC-AUC:   0.999

Classification Report:

              precision    recall  f1-score   support

           0       0.99      1.00      1.00       936
           1       0.98      0.97      0.98       190

    accuracy                           0.99      1126
   macro avg       0.99      0.98      0.99      1126
weighted avg       0.99      0.99      0.99      1126

Confusion Matrix:
[[932   4]
 [  5 185]]


In [26]:
# ------------------------------------------------------------
# 8. SAVE THE FINAL PIPELINE
# ------------------------------------------------------------
joblib.dump(final_pipeline, '../models/final_churn_model_pipeline.pkl')
print("\nModel pipeline saved as 'final_churn_model_pipeline.pkl'")


Model pipeline saved as 'final_churn_model_pipeline.pkl'
